# Correlazioni dinamiche del trimero a catena — misura via circuito, preparazione VQE reale

Mirror di `circuito_correlazioni_trimero_anello_vqe.ipynb`: stessa analisi del notebook
precedente, con la preparazione dello stato sostituita ovunque dal circuito VQE reale
(ansatz `W-2qC.K2`, 10 parametri) al posto di `prepare_state`.

Punto di lavoro: $J=1, b=b_c=3.0, D=0.15$ (VQE-DM), parametri già ottimizzati e salvati in
`w2qC_k2_params_vqedm.npz` (§0 li ricarica, non li ricalcola: l'ottimizzazione a 60 restart
è stata fatta in `vqe_w2qC_k2_trimero_catena.py`).

## 0. Setup e caricamento dei parametri VQE

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from trimer_chain_exact import trimer_hamiltonian_dm
from circuito_correlazioni_trimero_catena import ground_state, correlator_from_circuit
from validate_circuito_correlazioni_catena import classical_exact
from vqe_w2qC_k2_trimero_catena import bound_statevector

J, b, D = 1.0, 3.0, 0.15
H = trimer_hamiltonian_dm(J, b, D).to_matrix()
psi0_exact, E = ground_state(J, b, D)

data = np.load("w2qC_k2_params_vqedm.npz")
params = data["params"]
fidelity = float(data["fidelity"])
print(f"fidelity ansatz W-2qC.K2 = {fidelity:.14f}")

psi_vqe = bound_statevector(params)
print(f"|<psi0|psi_vqe>|^2 ricalcolato = {abs(np.vdot(psi0_exact, psi_vqe))**2:.14f}")

sites = (1, 2, 3)
comps = ('x', 'y', 'z')
labels = [f"{i}{a}" for i in sites for a in comps]

## 1. Misura diretta di tutte le 81 combinazioni ($t=1.3$, $N=200$)

In [ ]:
t_fix = 1.3
N = 200
rows = []
for i, al in itertools.product(sites, comps):
    for j, be in itertools.product(sites, comps):
        c_exact_prep = correlator_from_circuit(i, al, j, be, t_fix, N, J, b, D, psi0_exact)
        c_vqe = correlator_from_circuit(i, al, j, be, t_fix, N, J, b, D, ansatz_params=params)
        rows.append(dict(i=i, al=al, j=j, be=be, c_exact_prep=c_exact_prep, c_vqe=c_vqe,
                          residuo=abs(c_vqe - c_exact_prep)))
df = pd.DataFrame(rows)
print(f"combinazioni misurate: {len(df)}")

## 2. Validazione rapida

In [ ]:
print(f"Residuo (circuito VQE vs preparazione esatta): media={df['residuo'].mean():.3e}  "
      f"max={df['residuo'].max():.3e}")
print(f"riferimento sqrt(1-F) = {np.sqrt(max(0, 1-fidelity)):.3e}")

**Discussione.** Il residuo qui somma due contributi: errore di Trotter ($N=200$,
$\sim10^{-3}$) ed errore di preparazione VQE ($\sqrt{1-\mathcal F}$, qui $\sim10^{-7}$
essendo $\mathcal F\approx1$) — il primo domina nettamente, il secondo è trascurabile a
questo livello di fidelity.

## 3. Heatmap d'insieme

In [ ]:
idx = {lab: k for k, lab in enumerate(labels)}
n = len(labels)
M = np.zeros((n, n), dtype=complex)
for row in rows:
    M[idx[f"{row['i']}{row['al']}"], idx[f"{row['j']}{row['be']}"]] = row['c_vqe']

fig, ax = plt.subplots(figsize=(6, 5.3))
im = ax.imshow(np.abs(M), cmap='viridis', vmin=0, vmax=np.abs(M).max())
ax.set_xticks(range(n)); ax.set_yticks(range(n))
pretty = [f"${lab[0]}{lab[1]}$" for lab in labels]
ax.set_xticklabels(pretty, fontsize=9); ax.set_yticklabels(pretty, fontsize=9)
ax.set_xlabel(r'$(j,\beta)$ a $t=0$'); ax.set_ylabel(r'$(i,\alpha)$ a $t$')
ax.set_title(r'$|C_{ij}^{\alpha\beta}(t{=}1.3)|$, catena, preparazione VQE reale')
for k in (2.5, 5.5):
    ax.axhline(k, color='white', lw=0.8); ax.axvline(k, color='white', lw=0.8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
plt.show()

**Discussione.** Identica, a occhio, alla heatmap del notebook precedente (preparazione
esatta): i quadranti ricchi e quelli meno intensi sono nella stessa posizione.

## 4. Vista d'insieme nel tempo (griglia $9\times9$)

In [ ]:
t_grid = np.linspace(0.3, 10, 8)
fig, axes = plt.subplots(9, 9, figsize=(9, 9), dpi=60, sharex=True)
for r, (i, al) in enumerate(itertools.product(sites, comps)):
    for c, (j, be) in enumerate(itertools.product(sites, comps)):
        vals = [correlator_from_circuit(i, al, j, be, t, 60, J, b, D,
                                         ansatz_params=params).real for t in t_grid]
        ax = axes[r, c]
        ax.plot(t_grid, vals, lw=0.8, color='#8c1f4e')
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_ylim(-1, 1)
for r, (i, al) in enumerate(itertools.product(sites, comps)):
    axes[r, 0].set_ylabel(f"{i}{al}", fontsize=7, rotation=0, ha='right', va='center')
for c, (j, be) in enumerate(itertools.product(sites, comps)):
    axes[-1, c].set_xlabel(f"{j}{be}", fontsize=7)
fig.suptitle(r"$\mathrm{Re}\,C_{ij}^{\alpha\beta}(t)$, preparazione VQE, griglia $9\times9$", y=1.0)
fig.tight_layout()
plt.show()

**Lettura della griglia.** Identica, a occhio, a quella della versione a stato esatto
(notebook precedente): i quadranti ricchi restano ricchi, nessun pannello piatto a zero.

## 5. Selettore per ispezione singola

In [ ]:
def plot_correlatore_vqe(i, alpha, j, beta, t_max=10, N=200):
    ts = np.linspace(0.05, t_max, 60)
    vals = [correlator_from_circuit(i, alpha, j, beta, t, N, J, b, D, ansatz_params=params)
            for t in ts]
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.plot(ts, [v.real for v in vals], label='Re')
    ax.plot(ts, [v.imag for v in vals], label='Im')
    ax.set_xlabel('t'); ax.set_title(f"$C_{{{i}{j}}}^{{{alpha}{beta}}}(t)$, VQE")
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

plot_correlatore_vqe(2, "z", 2, "z")

## 6. Analisi: escursione, simmetria, spettro

In [ ]:
maxdiff = 0.0
for al in comps:
    for be in comps:
        c11 = correlator_from_circuit(1, al, 1, be, 1.3, 200, J, b, D, ansatz_params=params)
        c33 = correlator_from_circuit(3, al, 3, be, 1.3, 200, J, b, D, ansatz_params=params)
        maxdiff = max(maxdiff, abs(c11 - c33))
print(f"max|C_11 - C_33| su tutte le 9 componenti (via circuito VQE): {maxdiff:.2e}")
print("(entro l'errore di Trotter+VQE, coerente con lambda_alpha=+1 per ogni alpha)")

## 7. Confronto con la validazione a shot finiti (risultati precomputati, prep. VQE)

In [ ]:
d = np.load("scan81_catena_vqedm_results.npz")
err_trotter_mean = float(d['err_trotter_mean'])
err_shot_mean = float(d['err_shot_mean'])
print(f"Errore Trotter (precomputato, prep. esatta): media={err_trotter_mean:.2e}")
print(f"Errore statistico (precomputato): media={err_shot_mean:.2e}")
print(f"\nResiduo VQE osservato in questo notebook: media={df['residuo'].mean():.2e}")

**Discussione.** Numeri indistinguibili, alla precisione riportata, da quelli della
versione a stato esatto: la fidelity $\mathcal F\approx1$ rende il contributo della
preparazione VQE trascurabile rispetto all'errore di Trotter, qui come nell'anello.

## 8. Riepilogo

- La pipeline VQE $\to$ correlazioni è chiusa anche per la catena: il circuito con
  preparazione reale (`W-2qC.K2`) riproduce la versione a stato esatto entro un residuo
  dominato dall'errore di Trotter, non dalla preparazione.
- Heatmap, griglia temporale e relazione $P_{13}$ sono, a vista, indistinguibili dalla
  versione a preparazione esatta.

Con questo si chiude il mirror computazionale a quattro notebook della fase correlazioni
per la catena aperta.